In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC SUMMARY (Updated with Flattening & Bridging Architecture):
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double to find contracts >= 1,000,000.
   3. EXCEPTION HANDLING: Uses a 'Placeholder Swap' ([COMMA]) to protect known unit 
      names that contain commas (e.g., 'CAD PB - RESL...') from being improperly split.
   4. FLATTEN LOBs: Columns K and L can contain comma-separated lists. Uses LATERAL 
      VIEW explode(split(...)) to expand these strings into individual evaluation rows.
   5. RESTORE LOBs: Swaps '[COMMA]' back to a real comma after the explosion to match 
      the mapping table perfectly.
   6. TPRM STRING MAPPING & NAME BRIDGE: Maps expanded LOBs to AU IDs by checking if 
      the 'TPRM_Units' string exists inside the parsed LOB strings using SAFE LIKE 
      syntax (replacing the old RLIKE \b logic). Then bridges the Mapping Table's 
      String Name back to the Master List's Numeric BusinessID.
   7. AGGREGATING (NUMERATOR & DENOMINATOR): Counts total DISTINCT engagements 
      (Denominator) and total DISTINCT engagements >= 1MM (Numerator). DISTINCT is 
      crucial to avoid double-counting the flattened rows.
   8. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- Grab valid TPRM Mapping Strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units)

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - High Value Traceability with Comma Exceptions
   
   PURPOSE: Verifies Contract Amount cleaning and shows how the Exception 
   Dictionary safely protected comma-containing LOBs during the flattening process.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    SELECT 
        p.EngagementNumber,
        p.Contract_Amount AS Raw_Contract_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(p.Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        
        -- The original untouched strings
        p.owning_lob AS Full_Col_K,
        p.lob_using AS Full_Col_L,
        
        -- Exception Dictionary applied
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using,